In [ ]:
%load_ext autoreload
%autoreload 2

import umap
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from trace_comparison.triplet_accuracy import triplet_accuracy, relatedness_levels
from trace_comparison.data.dataset import load_trace_dataset
from trace_comparison.blocks.resample.resample import Resample
from trace_comparison.blocks.model_latent import LatentBlock
from utils.notebook_utils import reset_cwd_and_env

reset_cwd_and_env()
base_path = Path(os.path.join(os.getcwd()))

In [ ]:
dataset = load_trace_dataset(
    base_path / "data" / "traces_dataset" / "nonograms" / "base_1000.jsonl")

In [ ]:
data = dataset.traces
solvers = dataset.solvers
resample_block = Resample(k=20)
clf_hidden_block = ClassifierHidden(
    checkpoint_dir=base_path / "checkpoints",
    model_version="03",
    device="cpu"
    )
print(f"Data shape: {data.shape}")
data = resample_block(data)
print(f"Data shape: {data.shape}")
data = clf_hidden_block(data)
# data = data.reshape(data.shape[0], data.shape[1], -1)
print(f"Data shape: {data.shape}")
# minus each embedding by the puzzle mean embedding
puzzle_means = data.mean(axis=0)
print(f"Puzzle means shape: {puzzle_means.shape}")
data = data - puzzle_means
print(f"Data shape: {data.shape}")

In [ ]:
print(solvers)

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import umap
# from sklearn.decomposition import PCA
# import torch.nn.functional as F

# n_labels, n_samples, n_features = data.shape

# X = data.reshape(-1, n_features)           # (n_labels * n_samples, n_features)
# # Normalise each embedding using NumPy
# norms = np.linalg.norm(X, axis=1, keepdims=True)
# X = X / np.clip(norms, a_min=1e-12, a_max=None)
# y = np.repeat(np.arange(n_labels), n_samples)   # matches the reshape order

# # Optional: standardise / pre-reduce
# X50 = PCA(n_components=min(50, n_features)).fit_transform(X)

# # 2D projection
# X2 = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine",
#                random_state=0).fit_transform(X50)

# # Plot
# fig, ax = plt.subplots(figsize=(8, 8))
# cmap = plt.get_cmap("tab10" if n_labels <= 10 else "tab20")
# for i in range(n_labels):
#     pts = X2[y == i]
#     ax.scatter(pts[:, 0], pts[:, 1], s=6, alpha=0.6, color=cmap(i % cmap.N), label=f"label {i}")
#     ax.scatter(*pts.mean(0), s=120, color=cmap(i % cmap.N), edgecolor="k", marker="X")  # centroid
# ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
# ax.set_title("UMAP of embeddings coloured by label")
# plt.tight_layout()
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import umap

n_labels, n_samples, n_features = data.shape

X = data.reshape(-1, n_features)
X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)  # unit norm
y = np.repeat(np.arange(n_labels), n_samples)

X50 = PCA(n_components=min(100, n_features), random_state=0).fit_transform(X)

# Global structure: larger neighborhood, larger min_dist -> fewer artificial splits.
# After L2-normalisation euclidean ~ cosine, and much faster.
X2 = umap.UMAP(n_neighbors=20, min_dist=0.2, metric="euclidean",
               random_state=0).fit_transform(X50)

# Distinct color per class — no cycling (turbo / gist_ncar / viridis all fine)
colors = plt.get_cmap("turbo")(np.linspace(0.05, 0.95, n_labels))

# ---- overview panel -------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 8))
s = max(2, min(8, int(4000 / len(X))))          # scale point size with density
alpha = max(0.15, min(0.6, 50 / np.sqrt(len(X))))
for i in range(n_labels):
    pts = X2[y == i]
    ax.scatter(pts[:, 0], pts[:, 1], s=s, alpha=alpha, color=colors[i],
               linewidths=0, label=solvers[i])
    ax.scatter(*np.median(pts, axis=0), s=90, color=colors[i],
               edgecolor="k", marker="X", zorder=5)   # median, not mean (robust)
ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_title("UMAP of embeddings")
plt.tight_layout()
plt.show()

# ---- small multiples: one panel per class ---------------------------------
ncols = 6
nrows = int(np.ceil(n_labels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.4 * nrows),
                         sharex=True, sharey=True)
axes = np.atleast_2d(axes)
for r in range(nrows):
    for c in range(ncols):
        axes[r, c].scatter(X2[:, 0], X2[:, 1], s=1, c="lightgrey",
                           alpha=0.4, linewidths=0, rasterized=True)
for i in range(n_labels):
    ax = axes.flat[i]
    pts = X2[y == i]
    ax.scatter(pts[:, 0], pts[:, 1], s=2, color=colors[i], alpha=0.8, linewidths=0)
    ax.set_title(f"{solvers[i]}", fontsize=9)
for ax in axes.flat[n_labels:]:
    ax.axis("off")
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Per-class view (grey = all other classes)", y=1.0)
plt.tight_layout()
plt.show()

from sklearn.manifold import trustworthiness
print("trustworthiness:", trustworthiness(X50, X2, n_neighbors=15))

In [ ]:
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt

def linear_separability_score(X1, X2):
    """
    Returns Fisher's discriminant ratio.
    0 = identical distributions, →∞ = perfectly separable
    """
    n1, n2 = len(X1), len(X2)
    
    # Within-class scatter (trace of covariance)
    S_w = (n1 * np.cov(X1.T) + n2 * np.cov(X2.T)) / (n1 + n2)
    trace_w = np.trace(S_w)
    
    if trace_w == 0:
        return float('inf')  # All points identical within classes
    
    # Between-class scatter
    diff = X1.mean(axis=0) - X2.mean(axis=0)
    S_b = np.outer(diff, diff)
    
    return np.trace(S_b) / trace_w

pointclouds = [X2[y == i] for i in range(n_labels)]
similarity_scores = np.ones((n_labels, n_labels))
for (i, j) in combinations(range(n_labels), 2):
    score = linear_separability_score(pointclouds[i], pointclouds[j])
    similarity_scores[i, j] = 1 - score
    similarity_scores[j, i] = 1 - score  # Symmetric
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(similarity_scores, cmap='viridis', interpolation='nearest')
# Show all ticks and label them with the respective list entries
ax.set_xticks(np.arange(n_labels))
ax.set_yticks(np.arange(n_labels))
ax.set_xticklabels(solvers)
ax.set_yticklabels(solvers)


In [ ]:
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

def normalize_cloud(P):
    P = P.astype(np.float64)
    P = P - P.mean(0)                              # translation
    s = np.sqrt((P**2).sum(1).mean())              # RMS radius
    return P / max(s, 1e-12)                        # scale

def chamfer(A, B):
    """Symmetric squared Chamfer, mean over both directions."""
    dA, _ = cKDTree(B).query(A)
    dB, _ = cKDTree(A).query(B)
    return 0.5 * ((dA**2).mean() + (dB**2).mean())

def fscore(A, B, tau):
    """F-score at absolute distance tau (use after normalisation, e.g. tau=0.05)."""
    dA, _ = cKDTree(B).query(A)
    dB, _ = cKDTree(A).query(B)
    p = (dA < tau).mean()
    r = (dB < tau).mean()
    return 2 * p * r / max(p + r, 1e-12)

In [ ]:
# If n_features == 3, each row is xyz. If you also stored colour (e.g. n_features==6),
# restrict to the xyz columns — otherwise the RGB channels dominate the distance.
xyz = slice(0, 3) if n_features >= 3 else slice(0, n_features)

clouds = [normalize_cloud(data[i][:, xyz]) for i in range(n_labels)]

D = np.zeros((n_labels, n_labels))      # Chamfer distance
F = np.zeros((n_labels, n_labels))      # F-score (higher = more similar)
tau = 0.05                              # 5% of RMS radius — tune to your scale
for i in range(n_labels):
    for j in range(i + 1, n_labels):
        D[i, j] = D[j, i] = chamfer(clouds[i], clouds[j])
        F[i, j] = F[j, i] = fscore(clouds[i], clouds[j], tau)

# Convert distance -> similarity in (0,1] for downstream use
S = 1.0 / (1.0 + D)
np.fill_diagonal(S, 1.0)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(D, cmap="magma_r")   # darker = more dissimilar
ax.set_xticks(range(n_labels)); ax.set_yticks(range(n_labels))
ax.set_xticklabels(solvers, rotation=90, fontsize=8)
ax.set_yticklabels(solvers, fontsize=8)
ax.set_title("Raw Chamfer distance (darker = less similar)")
fig.colorbar(im, ax=ax, label="Chamfer distance")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

def _chamfer(A, B):
    dA,_ = cKDTree(B).query(A); dB,_ = cKDTree(A).query(B)
    return 0.5 * ((dA**2).mean() + (dB**2).mean())

def _fscore(A, B, tau):
    dA,_ = cKDTree(B).query(A); dB,_ = cKDTree(A).query(B)
    p = (dA < tau).mean(); r = (dB < tau).mean()
    return 2*p*r / max(p + r, 1e-12)

def split_clouds(Z, n_labels, n_samples, normalize=True):
    """Z: (n_labels*n_samples, d) flat array in row-major class order."""
    assert len(Z) == n_labels * n_samples, (len(Z), n_labels*n_samples)
    out = []
    for i in range(n_labels):
        P = np.asarray(Z[i*n_samples:(i+1)*n_samples], dtype=np.float64)
        if normalize:                       # remove per-class offset + size
            P = P - P.mean(0)
            P = P / max(np.sqrt((P**2).sum(1).mean()), 1e-12)
        out.append(P)
    return out

def pairwise_matrix(Z, n_labels, n_samples, metric="chamfer",
                    normalize=True, tau=0.05):
    clouds = split_clouds(Z, n_labels, n_samples, normalize=normalize)
    M = np.zeros((n_labels, n_labels))
    for i in range(n_labels):
        for j in range(i + 1, n_labels):
            if metric == "chamfer":
                v = _chamfer(clouds[i], clouds[j])
            else:
                v = _fscore(clouds[i], clouds[j], tau)
            M[i, j] = M[j, i] = v
    np.fill_diagonal(M, 0.0 if metric == "chamfer" else 1.0)
    return M

In [ ]:
# raw xyz (assuming the first 3 channels are coordinates)
Xxyz = data[..., :3].reshape(-1, 3)

# PCA: your X50 is already (n_labels*n_samples, 100) in class order.
# For Chamfer, 100-D KD-trees suffer from distance concentration — the first
# ~10-20 PCs carry almost all of it, so slice them off (PCA output is sorted).
Xpca  = X50[:, :20]

# UMAP: 2-D, already in class order (UMAP preserves row order)
Xumb  = X2

D_xyz  = pairwise_matrix(Xxyz, n_labels, n_samples, "chamfer", normalize=True)
D_pca  = pairwise_matrix(Xpca, n_labels, n_samples, "chamfer", normalize=True)
D_umb  = pairwise_matrix(Xumb, n_labels, n_samples, "chamfer", normalize=True)

In [ ]:
def show(D, title, ax):
    im = ax.imshow(D, cmap="magma_r", vmin=0, vmax=D.max())
    ax.set_xticks(range(n_labels)); ax.set_yticks(range(n_labels))
    ax.set_xticklabels(solvers, rotation=90, fontsize=7)
    ax.set_yticklabels(solvers, fontsize=7)
    off = D[np.triu_indices(n_labels, 1)]
    ax.set_title(f"{title}\nmin={off.min():.4f}  med={np.median(off):.4f}  "
                 f"max={off.max():.4f}", fontsize=9)
    return im

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for D, t, ax in zip([D_xyz, D_pca, D_umb],
                    ["xyz  (Chamfer)", "PCA-10  (Chamfer)", "UMAP-2  (Chamfer)"],
                    axes):
    im = show(D, t, ax)
    fig.colorbar(im, ax=ax, fraction=0.046, label="distance")
plt.tight_layout(); plt.show()

# quick numeric check of dynamic range per space
for name, D in [("xyz", D_xyz), ("pca", D_pca), ("umap", D_umb)]:
    off = D[np.triu_indices(n_labels, 1)]
    print(f"{name:5s}: min={off.min():.4f} med={np.median(off):.4f} "
          f"max={off.max():.4f}  spread={off.max()-off.min():.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import umap

n_labels, n_samples, n_features = data.shape
X = data.reshape(-1, n_features).astype(np.float64)
y = np.repeat(np.arange(n_labels), n_samples)

# --- 1. centre THEN normalise (removes the shared "cone" direction) ---------
X = X - X.mean(axis=0, keepdims=True)
X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)

X50 = PCA(n_components=min(100, n_features), random_state=0).fit_transform(X)

# --- 2. class-level distances in the ORIGINAL space -------------------------
centroids = np.stack([X[y == i].mean(0) for i in range(n_labels)])
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
D = 1 - centroids @ centroids.T                       # cosine distance
np.fill_diagonal(D, 0)

# within-class spread: median distance of members to their own centroid
r = np.array([np.median(np.linalg.norm(X[y == i] - X[y == i].mean(0), axis=1))
              for i in range(n_labels)])
cdist = np.linalg.norm(centroids[:, None] - centroids[None], axis=-1)
sep = cdist / (0.5 * (r[:, None] + r[None]))          # >~1 => centres further apart than the spread
np.fill_diagonal(sep, np.nan)

order = leaves_list(linkage(squareform(D, checks=False), "average"))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, M, title in [(axes[0], D, "Centroid cosine distance"),
                     (axes[1], sep, "Separation = centroid dist / within-class spread")]:
    im = ax.imshow(M[np.ix_(order, order)], cmap="viridis")
    ax.set_xticks(range(n_labels)); ax.set_yticks(range(n_labels))
    ax.set_xticklabels(np.array(solvers)[order], rotation=90, fontsize=7)
    ax.set_yticklabels(np.array(solvers)[order], fontsize=7)
    ax.set_title(title); plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

# --- 3. honest separability numbers -----------------------------------------
print("silhouette (cosine):", silhouette_score(X50, y, metric="cosine"))
lda_cv = LDA(solver="eigen", shrinkage="auto")
print("5-fold LDA accuracy:", cross_val_score(lda_cv, X50, y, cv=5).mean(),
      "| chance:", 1 / n_labels)

# --- 4. projections that keep distances meaningful --------------------------
lda = LDA(solver="eigen", shrinkage="auto", n_components=2).fit(X50, y)
X_lda = lda.transform(X50)                            # max class separation
X_pca = X50[:, :2]                                    # linear, distance-preserving

# UMAP tuned toward global layout: PCA init, small min_dist, bigger neighbourhood
X_umap = umap.UMAP(n_neighbors=50, min_dist=0.05, spread=1.0, init="pca",
                   metric="euclidean", random_state=0).fit_transform(X50)

# --- 5. plotting helper: zoom on the bulk, not the outliers -----------------
colors = plt.get_cmap("turbo")(np.linspace(0.05, 0.95, n_labels))

def plot(ax, Z, title):
    for i in range(n_labels):
        p = Z[y == i]
        ax.scatter(p[:, 0], p[:, 1], s=3, alpha=0.4, color=colors[i], linewidths=0)
        ax.scatter(*np.median(p, 0), s=80, color=colors[i], edgecolor="k", marker="X", zorder=5)
    lo, hi = np.percentile(Z, [1, 99], axis=0)        # clip axes to central 98%
    pad = 0.1 * (hi - lo)
    ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0]); ax.set_ylim(lo[1] - pad[1], hi[1] + pad[1])
    ax.set_title(title); ax.set_aspect("equal", "datalim")

fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
plot(axes[0], X_pca, "PCA (distances meaningful)")
plot(axes[1], X_lda, "LDA (what a linear classifier can separate)")
plot(axes[2], X_umap, "UMAP (shape only)")
axes[2].legend(solvers, markerscale=3, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
P = data.mean(axis=1)                                  # (n_labels, D) solver prototypes
P = P / np.clip(np.linalg.norm(P, axis=1, keepdims=True), 1e-12, None)
Sim = P @ P.T                                        # cosine similarity matrix

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
order = dendrogram(linkage(squareform(1 - Sim, checks=False), method="average"),
                   no_plot=True)["leaves"]

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(Sim[np.ix_(order, order)], cmap="viridis", vmin=0.9, vmax=1)
ax.set_xticks(range(n_labels), [str(solvers[i]) for i in order], rotation=90, fontsize=7)
ax.set_yticks(range(n_labels), [str(solvers[i]) for i in order], fontsize=7)
fig.colorbar(im, label="prototype cosine similarity")
ax.set_title("Solver × solver similarity (hierarchically ordered)")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# emb: (n_labels * n_samples, hidden) hidden-layer activations, y: matching labels
emb = data.reshape(-1, data.shape[-1])
y = np.repeat(np.arange(data.shape[0]), data.shape[1])
knn = KNeighborsClassifier(n_neighbors=5, metric="cosine")
scores = cross_val_score(knn, emb, y, cv=5)
print(f"5-NN accuracy in embedding space: {scores.mean():.3f}")

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import silhouette_score

X = data.reshape(-1, data.shape[-1])
y = np.repeat(np.arange(data.shape[0]), data.shape[1])

cv = StratifiedKFold(5, shuffle=True, random_state=0)
lin = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(10, metric="cosine"))

print("chance:        ", 1 / data.shape[0])
print("linear probe:  ", cross_val_score(lin, X, y, cv=cv).mean())
print("kNN:           ", cross_val_score(knn, X, y, cv=cv).mean())
print("silhouette:    ", silhouette_score(X, y, metric="cosine", sample_size=5000, random_state=0))

# Between-class vs within-class variance
mu = X.mean(0)
class_means = np.stack([X[y == k].mean(0) for k in range(data.shape[0])])
between = ((class_means - mu) ** 2).sum(1).mean()
within = np.mean([((X[y == k] - class_means[k]) ** 2).sum(1).mean() for k in range(data.shape[0])])
print("between/within:", between / within)